# DNS Resolver Comparison Analysis

Demonstrates loading DNS data and comparing resolver performance, stability, and consistency.

In [ ]:
import sys
sys.path.insert(0, '..')

from inventor_analysis.loaders import load_dns_data
from inventor_analysis.dns_analysis import (
    compute_resolver_statistics,
    compute_resolver_stability,
    detect_dns_anomalies,
    compute_threshold_breaches,
    compute_resolver_consistency,
    rank_resolvers,
)

## Load Data

Point at a directory with Inventor DNS JSONL files. Uses `sample_data/` by default.

In [ ]:
df = load_dns_data("../sample_data", max_files=5)

print(f"Loaded {len(df):,} records")
print(f"Resolvers: {df['resolver'].unique()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## Resolver Performance Statistics

Per-resolver response time aggregation and success rate.

In [ ]:
stats = compute_resolver_statistics(df)
stats

## Resolver Stability (Coefficient of Variation)

Lower CV = more consistent response times.

In [ ]:
stability = compute_resolver_stability(df)
stability

## Threshold Breaches

Fraction of queries exceeding 20ms response time.

In [ ]:
breaches = compute_threshold_breaches(df, threshold_ms=20)
breaches

## DNS Anomaly Detection (Z-Score)

In [ ]:
df_anomalies = detect_dns_anomalies(df, threshold=3.0)
anomalies = df_anomalies[df_anomalies['is_anomaly']]

print(f"Total anomalies: {len(anomalies):,}")

if not anomalies.empty:
    print("\nAnomaly breakdown by resolver:")
    display(anomalies.groupby('resolver').size())

## IP Set Consistency (Jaccard Similarity)

Measures how stable the returned IP sets are across days. A Jaccard similarity of 1.0 means the resolver returned the same IPs every day.

In [ ]:
try:
    consistency = compute_resolver_consistency(df)
    display(consistency[['resolver', 'file_date', 'unique_ips', 'jaccard_prev']])
except Exception as e:
    print(f"Consistency analysis skipped ({e})")

## Overall Resolver Rankings

Combined ranking across speed, stability, and reliability.

In [ ]:
rankings = rank_resolvers(df)
rankings